In [8]:
# --- KNN (choix du nombre de voisins) ---
from sklearn.neighbors import KNeighborsClassifier           # l'algorithme KNN
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score  # validation croisée
from sklearn.pipeline import make_pipeline                    # enchaîne plusieurs étapes en un seul modèle
from sklearn.preprocessing import StandardScaler              # met les variables à la même échelle
import pandas as pd                                           # lecture et manipulation du tableau

# Lecture du fichier des abonnés 
ab = pd.read_csv("../data/processed/abonnes_annuels_clean.csv")

# On ne garde que les abonnés annuels : la question porte sur eux
an = ab[ab["formule"] == "annuel"].copy()

# Cible : 1 = n'a pas renouvelé (churn), 0 = a renouvelé
an["churn"] = (an["a_renouvele"] == "non").astype(int)

# X = les 3 signaux qu'on utilise pour prédire ; y = ce qu'on veut prédire
X = an[["anciennete_mois", "trajets_par_mois", "incidents_signales"]]
y = an["churn"]

# Validation croisée : on coupe les données en 5 morceaux, on entraîne sur 4,
# on teste sur le 5e, à tour de rôle ; on répète 10 fois avec des découpages
# différents = 50 tests. "Stratifiée" : chaque morceau garde 22 % de partants.
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

# On essaie plusieurs nombres de voisins k et on compare leur qualité
for k in [5, 15, 25, 35, 49, 65, 85]:
    # Le modèle = standardisation puis KNN avec k voisins.
    # La standardisation est indispensable : KNN calcule des distances, et sans
    # elle l'ancienneté (1 à 60) écraserait les incidents (0 à 3).
    modele = make_pipeline(StandardScaler(), KNeighborsClassifier(k))
    # AUC sur les 50 tests (1 = parfait, 0,5 = hasard)
    s = cross_val_score(modele, X, y, cv=cv, scoring="roc_auc")
    print(f"k={k:>3} : AUC {s.mean():.3f}")                   # moyenne des 50 AUC

k=  5 : AUC 0.531
k= 15 : AUC 0.531
k= 25 : AUC 0.577
k= 35 : AUC 0.610
k= 49 : AUC 0.633
k= 65 : AUC 0.619
k= 85 : AUC 0.633


In [9]:
# --- K-means : choix du nombre de profils et taux de départ par profil ---
from sklearn.cluster import KMeans                 # l'algorithme K-means
from sklearn.metrics import silhouette_score       # mesure la qualité d'un regroupement

# Standardisation des 3 signaux : K-means calcule aussi des distances
Z = StandardScaler().fit_transform(X)

# On essaie de 2 à 8 groupes et on mesure la silhouette de chaque découpage
for k in range(2, 9):
    # n_init=10 : K-means part de 10 positions de départ et garde la meilleure ;
    # random_state=42 : résultat reproductible
    km = KMeans(k, n_init=10, random_state=42).fit(Z)
    # Silhouette (de -1 à 1) : plus elle est haute, plus les groupes sont
    # compacts et bien séparés les uns des autres
    print(f"k={k} : silhouette {silhouette_score(Z, km.labels_):.3f}")

# On retient 4 profils et on attribue un numéro de profil à chaque abonné
an["profil"] = KMeans(4, n_init=10, random_state=42).fit_predict(Z)

# Pour chaque profil : taux de départ (moyenne de churn) et moyenne des 3 signaux
print(an.groupby("profil")[["churn", "anciennete_mois", "trajets_par_mois",
                            "incidents_signales"]].mean().round(2))

k=2 : silhouette 0.322
k=3 : silhouette 0.334
k=4 : silhouette 0.362
k=5 : silhouette 0.362
k=6 : silhouette 0.348
k=7 : silhouette 0.366
k=8 : silhouette 0.367
        churn  anciennete_mois  trajets_par_mois  incidents_signales
profil                                                              
0        0.34            16.68              7.03                1.28
1        0.22            16.10              7.15                0.00
2        0.12            43.69              7.49                0.34
3        0.09            20.52             20.96                0.34


In [10]:
# --- Régression linéaire : effet de chaque signal (points de %, par écart-type) ---
from sklearn.linear_model import LinearRegression   # l'algorithme de régression linéaire

# Standardisation puis régression linéaire, entraînée sur tous les annuels.
# Comme churn vaut 0 ou 1, la droite prédit une probabilité de départ.
linear = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)

# lin[-1] = la dernière étape du pipeline (la régression) ; coef_ = ses coefficients.
# Variables standardisées : chaque coefficient = effet de +1 écart-type de la
# variable sur la probabilité de départ. x 100 pour l'avoir en points de %.
for nom, c in zip(X.columns, linear[-1].coef_):
    print(f"{nom:<20} {c * 100:+.1f} pts")

anciennete_mois      -4.2 pts
trajets_par_mois     -6.1 pts
incidents_signales   +6.2 pts
